# Read an NDJSON file in Python with chDB (drop-in pandas)

Companion to [read-ndjson-file-python](https://clickhouse.com/resources/engineering/read-ndjson-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas`.

In [ ]:
import chdb.datastore as pd

## 1. Read an NDJSON file into a DataFrame (types auto-inferred)

NDJSON is one JSON object per line. `pd.read_json(..., lines=True)` reads it directly and infers the column types.

In [ ]:
df = pd.read_json("data/events.ndjson", lines=True)
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
purchases = df[df["event_type"] == "purchase"].groupby("country")["revenue"].sum().sort_values(ascending=False)
purchases

## 3. Access nested fields with pandas-style indexing

Nested object columns come back as lists or dicts. Use `.to_pandas()` and `.apply()` to extract sub-fields.

In [ ]:
user_tiers = df[["event_id", "revenue"]].copy().to_pandas()
user_tiers["tier"] = df["user"].to_pandas().apply(lambda u: u["tier"] if isinstance(u, dict) else u[1])
gold = user_tiers[user_tiers["tier"] == "gold"]
gold

## 4. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 5. Performance: same code, one import swapped, on a 2M-row NDJSON file

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~5.5x faster than `pandas.read_json(lines=True)`.

In [ ]:
import time

def datastore_agg():
    d = pd.read_json("data/events_large.ndjson", lines=True)
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["revenue"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pandas_agg():
    import pandas as real_pd
    p = real_pd.read_json("data/events_large.ndjson", lines=True)
    return (p[p["event_type"] == "purchase"]
            .groupby("country")["revenue"].sum()
            .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

pd_s = best_of_3(pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"import pandas as pd:            {pd_s:.3f}s")
print(f"import chdb.datastore as pd:    {ds_s:.3f}s")
print(f"speedup:                        {pd_s / ds_s:.1f}x")